```python
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import requests

# 1. Initialize the app
app = dash.Dash(__name__)

# 2. Data Fetching Logic
def get_data():
    url = "https://restcountries.com/v3.1/all?fields=name,region,population,area,cca3"
    response = requests.get(url)
    df = pd.json_normalize(response.json())
    return df.rename(columns={"name.common": "country"})

df = get_data()

# 3. App Layout
app.layout = html.Div([
    html.H1("Dash Global Dashboard", style={'textAlign': 'center'}),
    
    html.Div([
        html.Label("Selecciona una Región:"),
        dcc.Dropdown(
            id='region-dropdown',
            options=[{'label': r, 'value': r} for r in sorted(df['region'].unique())],
            value='Africa'  # Default value
        ),
    ], style={'width': '30%', 'padding': '20px'}),

    dcc.Graph(id='choropleth-map')
])

# 4. Callbacks (The "Brain")
@app.callback(
    Output('choropleth-map', 'figure'),
    Input('region-dropdown', 'value')
)
def update_map(selected_region):
    filtered_df = df[df['region'] == selected_region]
    
    fig = px.choropleth(
        filtered_df,
        locations="cca3",
        color="population",
        hover_name="country",
        scope=selected_region.lower() if selected_region.lower() in ['africa', 'europe', 'asia'] else 'world',
        color_continuous_scale="Viridis"
    )
    
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
    return fig

# 5. Run the server
if __name__ == '__main__':
    app.run_server(debug=True)
```